# 第一章：环境搭建与图数据库核心概念

## 学习目标

- 理解图数据库与关系型数据库的区别
- 使用 Docker 搭建 Neo4j 环境
- 掌握图的核心概念：节点、关系、属性、标签
- 使用 Python 驱动连接 Neo4j
- 完成第一个图数据操作

## 1. 为什么需要图数据库？

假设我们有一个社交网络，需要查询**"张三的朋友的朋友"**。

### 关系型数据库方案

需要 `users` 和 `friendships` 两张表，加上多重 JOIN：

```sql
SELECT u3.name
FROM users u1
JOIN friendships f1 ON u1.id = f1.user_id
JOIN friendships f2 ON f1.friend_id = f2.user_id
JOIN users u3 ON f2.friend_id = u3.id
WHERE u1.name = '张三' AND u3.id != u1.id;
```

表越多、层数越深，SQL 越复杂，性能也急剧下降。

### 图数据库方案

一句 Cypher 搞定：

```cypher
MATCH (p:Person {name: '张三'})-[:KNOWS*2]->(fof:Person)
WHERE fof <> p
RETURN fof.name
```

`[:KNOWS*2]` 表示沿 `KNOWS` 关系走两步——表达力和性能都远优于 JOIN。

### 对比总结

| | 关系型数据库 | 图数据库 |
|---|---|---|
| **数据模型** | 表（行和列） | 节点和关系 |
| **关联查询** | JOIN（性能随深度下降） | 遍历（性能稳定） |
| **Schema** | 固定（需预定义表结构） | 灵活（可随时添加属性） |
| **适合场景** | 结构化事务处理 | 高度关联数据（社交、知识图谱） |
| **查询语言** | SQL | Cypher |

## 2. 图的核心概念

图数据库围绕四个核心概念构建：

- **节点（Node）**：图中的实体（人、电影、公司）。可以有多个**标签（Label）**。
- **关系（Relationship）**：节点之间的有向连接（`KNOWS`、`ACTED_IN`、`DIRECTED`）。**总是有方向和类型**。
- **属性（Property）**：节点和关系都可以有键值对属性（`name: "张三"`, `age: 30`）。
- **标签（Label）**：节点的分类标记（`Person`、`Movie`）。一个节点可以有多个标签。

### 一图胜千言

```
(:Person {name:"张三"}) -[:KNOWS {since:2020}]-> (:Person {name:"李四"})
         |
         +-[:WORKS_AT]-> (:Company {name:"Acme"})
```

- 圆括号 `()` 代表节点，冒号后面是标签，花括号是属性
- 方括号 `[]` 代表关系，箭头表示方向
- 关系也可以有属性（如 `since: 2020`）

## 3. 使用 Docker 启动 Neo4j

在终端运行以下命令启动 Neo4j：

```bash
docker run -d \
  --name neo4j \
  -p 7474:7474 \
  -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/your_password \
  -e NEO4J_PLUGINS='["apoc"]' \
  neo4j:latest
```

启动后可以访问：
- **Neo4j Browser**：http://localhost:7474 （可视化界面，可以直接写 Cypher）
- **Bolt 协议**：`bolt://localhost:7687`（Python 驱动连接地址）

> Neo4j 启动需要几秒钟，如果连接失败请稍等一下再试。

### 配置环境变量

在项目根目录的 `.env` 文件中添加以下内容：

```
NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
```

> 密码要和 Docker 启动时 `NEO4J_AUTH` 中设置的一致。

## 4. Python 驱动连接

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)

# 验证连接
driver.verify_connectivity()
print("✓ Neo4j 连接成功")

In [ ]:
# 查看服务器信息
info = driver.get_server_info()
print(f"地址: {info.address}")
print(f"协议版本: {info.protocol_version}")

### 刚才发生了什么？

1. `GraphDatabase.driver()` 创建一个连接驱动，参数是 Bolt 地址和认证信息
2. `verify_connectivity()` 尝试与 Neo4j 建立连接，失败会抛异常
3. `get_server_info()` 返回服务器地址和协议版本，确认连接细节

> `driver` 对象是**线程安全**的，整个应用生命周期创建一次即可，不需要每次查询都新建。

## 5. 第一个图操作

In [ ]:
# 清空数据库（确保从零开始）
driver.execute_query("MATCH (n) DETACH DELETE n")
print("✓ 数据库已清空")

In [ ]:
# 创建两个 Person 节点和一个 KNOWS 关系
driver.execute_query("""
    CREATE (a:Person {name: '张三', age: 30})
    CREATE (b:Person {name: '李四', age: 25})
    CREATE (a)-[:KNOWS {since: 2020}]->(b)
    RETURN a, b
""")
print("✓ 已创建节点和关系")

In [ ]:
# 查询图中的所有 Person 节点
records, summary, keys = driver.execute_query(
    "MATCH (p:Person) RETURN p.name AS name, p.age AS age"
)

print("图中的 Person 节点：")
for record in records:
    print(f"  {record['name']}，{record['age']} 岁")

### 刚才发生了什么？

1. **`driver.execute_query()`** 向 Neo4j 发送 Cypher 语句，返回三元组 `(records, summary, keys)`：
   - `records`：查询结果列表，每个元素是一行数据
   - `summary`：执行摘要（包括计数、耗时等）
   - `keys`：返回的字段名列表
2. **`CREATE`** 创建节点（带标签和属性）和关系
3. **`MATCH`** 是查询模式——找出匹配指定结构的节点和关系
4. **`RETURN`** 指定输出内容（类似 SQL 的 `SELECT`）
5. **属性访问**：`p.name` 取出节点 `p` 的 `name` 属性

In [ ]:
# 扩展图：添加更多节点和关系
driver.execute_query("""
    MATCH (a:Person {name: '张三'})
    CREATE (c:Person {name: '王五', age: 28})
    CREATE (d:Company {name: 'Acme 科技', founded: 2015})
    CREATE (a)-[:KNOWS]->(c)
    CREATE (a)-[:WORKS_AT {role: '工程师'}]->(d)
    CREATE (c)-[:WORKS_AT {role: '设计师'}]->(d)
""")
print("✓ 已扩展图结构")

In [ ]:
# 查询"张三的朋友"
records, _, _ = driver.execute_query("""
    MATCH (p:Person {name: '张三'})-[:KNOWS]->(friend:Person)
    RETURN friend.name AS name
""")

print("张三的朋友：")
for r in records:
    print(f"  {r['name']}")

In [ ]:
# 查询"在同一家公司工作的人"
records, _, _ = driver.execute_query("""
    MATCH (p1:Person)-[:WORKS_AT]->(c:Company)<-[:WORKS_AT]-(p2:Person)
    WHERE p1 <> p2
    RETURN p1.name AS person1, p2.name AS person2, c.name AS company
""")

print("同事关系：")
for r in records:
    print(f"  {r['person1']} 和 {r['person2']} 都在 {r['company']}")

### 刚才发生了什么？

上面三段代码演示了图查询最大的优势：**关系遍历**。

- **"张三的朋友"**：从张三出发，沿 `KNOWS` 关系走一步，找到直接连接的 Person
- **"同事关系"**：两个 Person 都指向同一个 Company——这在图里是一个很自然的模式，但在关系型数据库里需要 JOIN 三张表

注意 Cypher 的模式语法和我们之前画的 ASCII 图几乎一模一样：
```
(p1:Person)-[:WORKS_AT]->(c:Company)<-[:WORKS_AT]-(p2:Person)
```
**查询语句就是数据结构的直接描述**，这是 Cypher 可读性强的关键原因。

## 6. 图数据库与向量检索的关系

在 LangChain 第五章中，我们学过基于向量的 RAG：把文档切块、嵌入向量空间、用语义相似度检索。图数据库提供了一种**完全不同的知识表示方式**——用结构化的节点和关系来组织信息。

| | 向量检索（RAG） | 图数据库 |
|---|---|---|
| **数据表示** | 扁平的文档块 | 节点和关系的网络 |
| **查询方式** | 语义相似度 | 结构化模式匹配 |
| **擅长** | "和 X 相关的内容" | "X 和 Y 之间的关系" |
| **已学章节** | LangChain 第五章 | 本教程 |

两者并不互斥——后续章节我们会学习 **GraphRAG**，把图数据库和向量检索结合起来，让 LLM 既能理解语义，又能推理关系。

### 常见问题

- **连接失败 `ServiceUnavailable`**：确认 Docker 容器在运行（`docker ps`），Neo4j 启动需要几秒，稍等再重试。
- **认证失败 `AuthError`**：检查 `.env` 中的密码是否与 Docker 启动时 `NEO4J_AUTH` 设置的一致。
- **`APOC` 插件未安装**：后续章节需要 APOC，确保 Docker 启动时加了 `-e NEO4J_PLUGINS='["apoc"]'`。
- **端口被占用**：如果 7474 或 7687 端口已被占用，可以换成其他端口，例如 `-p 17474:7474 -p 17687:7687`，同时更新 `.env` 中的 `NEO4J_URI`。

In [ ]:
# 清理（可选：删除所有数据）
driver.execute_query("MATCH (n) DETACH DELETE n")
print("✓ 数据已清理")

# 关闭驱动
driver.close()
print("✓ 连接已关闭")

## 总结

本章学习了：

- ✅ 图数据库与关系型数据库的核心区别
- ✅ 图的四要素：节点、关系、属性、标签
- ✅ Docker 一键启动 Neo4j
- ✅ Python 驱动连接和验证
- ✅ `CREATE` 创建节点和关系
- ✅ `MATCH` 查询图数据
- ✅ 图数据库与向量检索的互补关系

## 下一章预告

**第二章：Cypher 查询语言** —— 本章我们用简单的 `CREATE`/`MATCH` 操作了图。下一章系统学习 Cypher 语言：模式匹配、条件过滤、路径查询、更新删除等，掌握图的"SQL"。